# Day 17 — Practice Session · **SOLUTIONS**
### GroupBy & Aggregation · Python for Data Science

**Prepared by Srinivasa Sai Chava**  ·  Boston University

---

> **Instructor copy.** Every question is followed by the answer and the reasoning.
> The student copy (`Day17_Practice_Questions.ipynb`) is identical minus the answer blocks.

| Part | Focus | Questions |
|---|---|---|
| A | Predict the output | 8 |
| B | Spot & fix the bug | 4 |
| C | Write the code | 5 |
| D | Challenge | 2 |

---
## Setup — run this first

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.simplefilter("ignore")

sales = pd.DataFrame({
    "rep":     ["Ravi", "Sara", "Amit", "Neha", "Kiran", "Priya", "Dev", "Meena"],
    "region":  ["West", "West", "East", "East", "North", "North", "West", "East"],
    "product": ["A", "B", "A", "B", "A", "B", "B", "A"],
    "units":   [120, 95, 60, 140, 80, 110, 45, 130],
    "revenue": [24000, 19000, 12000, 28000, 16000, 22000, 9000, 26000],
})
print("pandas", pd.__version__)
sales

---
## Setup — run this first

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.simplefilter("ignore")

sales = pd.DataFrame({
    "rep":     ["Ravi", "Sara", "Amit", "Neha", "Kiran", "Priya", "Dev", "Meena"],
    "region":  ["West", "West", "East", "East", "North", "North", "West", "East"],
    "product": ["A", "B", "A", "B", "A", "B", "B", "A"],
    "units":   [120, 95, 60, 140, 80, 110, 45, 130],
    "revenue": [24000, 19000, 12000, 28000, 16000, 22000, 9000, 26000],
})
print("pandas", pd.__version__)
sales

---
# Part A — Predict the Output  *(8 min)*

---
# Part A — Predict the Output

**Teaching note:** A6 (size vs count) and A8 (the missing key) are the two that matter. A8
is the one that produces wrong reports in real work.

### A1. What comes back, and in what order?

In [ ]:
print(sales.groupby("region")["revenue"].sum())

*Your prediction:*  

> ### ✅ Answer A1
> ```
> region
> East      66000
> North     38000
> West      52000
> Name: revenue, dtype: int64
> ```
> **Why:** East = 12000 + 28000 + 26000, North = 16000 + 22000, West = 24000 + 19000 + 9000.
>
> **Note the order: alphabetical by group key, not by value.** West has the most reps but
> appears last. Students routinely read the first row as "the biggest" — it is not. To rank
> by value you must sort the result yourself.

### A2. What type is each of these?

In [ ]:
gb = sales.groupby("region")
print(type(gb).__name__)
print(type(sales.groupby("region")["revenue"].sum()).__name__)
print(type(sales.groupby("region")[["units", "revenue"]].sum()).__name__)

*Your prediction:*  

> ### ✅ Answer A2
> ```
> DataFrameGroupBy
> Series
> DataFrame
> ```
> **Why:** the groupby object on its own is **lazy** — nothing is computed until you ask for
> a statistic. Printing it shows an object, not a table, which surprises people.
>
> One column aggregated gives a **Series**; several columns give a **DataFrame**. Same
> single-versus-double bracket rule as Day 15.

### A3. What is the index of the result?

In [ ]:
r = sales.groupby("region")["units"].mean()
print(r.index.tolist())
print("region" in r.reset_index().columns)

*Your prediction:*  

> ### ✅ Answer A3
> ```
> ['East', 'North', 'West']
> True
> ```
> **Why:** the group key becomes the **index** of the result, not a column. That is why
> `result["region"]` raises a `KeyError` — a very common surprise.
>
> `reset_index()` turns it back into a column, and `groupby(..., as_index=False)` skips the
> index step entirely. Use either when the result is heading into a merge or a CSV.

### A4. How do these two differ?

In [ ]:
print(sales.groupby("region")["units"].agg(["mean", "max"]))
print()
print(sales.groupby("region").agg({"units": "mean", "revenue": "max"}))

*Your prediction:*  

> ### ✅ Answer A4
> ```
>          mean  max
> region
> East      110  140
> North      95  110
> West       ~86.67  120
>
>              units  revenue
> region
> East    110.000000    28000
> North    95.000000    22000
> West     86.666667    24000
> ```
> **Why:** a **list** applies several functions to the *same* column, producing one column
> per function. A **dictionary** applies a different function to each named column — Day 5's
> dictionary, telling pandas what to do with what.
>
> West's mean is 86.67 (120 + 95 + 45 = 260, over 3 reps).

### A5. What do the named results look like?

In [ ]:
print(sales.groupby("region").agg(
    reps       = ("rep", "size"),
    total_rev  = ("revenue", "sum"),
    best_units = ("units", "max"),
))

*Your prediction:*  

> ### ✅ Answer A5
> ```
>         reps  total_rev  best_units
> region
> East       3      66000         140
> North      2      38000         110
> West       3      52000         120
> ```
> **Why:** **named aggregation** — each output column gets the name *you* chose, in the form
> `new_name = (column, function)`.
>
> This is the version to default to. The columns read like a report rather than `size`,
> `sum`, `max`, and it avoids the two-level column headings that
> `agg(["mean", "max"])` creates across several columns.

### A6. Why do these disagree?

In [ ]:
d = sales.copy()
d.loc[2, "revenue"] = np.nan

print(d.groupby("region").size())
print()
print(d.groupby("region")["revenue"].count())

*Your prediction:*  

> ### ✅ Answer A6
> ```
> region
> East     3
> North    2
> West     3
>
> region
> East     2
> North    2
> West     3
> ```
> **Why:** `size()` counts **rows**. `count()` counts **non-missing values** in that column.
> Amit's revenue is now missing, so East drops from 3 to 2 in the second.
>
> **If those two disagree, you have missing data.** Reporting both is what makes an average
> honest — a mean over 2 of 3 rows is a different claim from a mean over all 3.

### A7. What shape does each produce?

In [ ]:
a = sales.groupby("region")["units"].mean()
b = sales.groupby("region")["units"].transform("mean")
print(len(a), len(b), len(sales))

*Your prediction:*  

> ### ✅ Answer A7
> ```
> 3 8 8
> ```
> **Why:** `agg` (here `.mean()`) **shrinks** the table to one row per group — three regions,
> three rows. `transform` keeps the **original shape**, repeating each group's answer for
> every member of that group.
>
> That is exactly why `transform` is what you need for a question like *"which reps beat
> their own region's average?"* — you need one value per **row**, aligned with the original
> table, so you can compare.

### A8. Do the numbers add up?

In [ ]:
d = sales.copy()
d.loc[0, "region"] = np.nan

counts = d.groupby("region").size()
print(counts)
print("total:", counts.sum(), "  rows in table:", len(d))

*Your prediction:*  

> ### ✅ Answer A8
> ```
> region
> East     3
> North    2
> West     2
> dtype: int64
> total: 7   rows in table: 8
> ```
> **Ravi vanished.** His region is missing, so he belongs to no group and is silently
> excluded — **no error, no warning**, and West quietly drops from 3 to 2.
>
> **Two defences:**
> - `groupby("region", dropna=False)` makes the missing group visible as `NaN`
> - compare `counts.sum()` against `len(df)` — the cheapest audit there is
>
> This is the **third** silent-missing-data trap of the week, after Day 14's `nansum`
> returning `0.0` and Day 15's pandas skipping `nan` in a mean. The pattern is always the
> same: the number looks reasonable and nothing complains.

---
# Part B — Spot & Fix the Bug  *(7 min)*

---
# Part B — Spot & Fix the Bug

**Teaching note:** B1 is the KeyError everyone hits at least once. B4 is the silent one.

### B1. This should print the region beside its total.

In [ ]:
totals = sales.groupby("region")["revenue"].sum()
print(totals["region"])

*Error:* 

*Your fix:*

> ### ✅ Answer B1
> **Error:** `KeyError: 'region'`
> **Cause:** after a groupby, the key is the **index**, not a column. There is no column
> called `region` in the result any more.

In [ ]:
totals = sales.groupby("region")["revenue"].sum()

print("the index IS the region:", totals.index.tolist())
print("look one up:", totals["West"])
print()

# To get it back as a column:
flat = totals.reset_index()
print(flat)
print(flat["region"].tolist())

# Or skip the index step entirely:
print()
print(sales.groupby("region", as_index=False)["revenue"].sum())

### B2. This should show the highest-earning region first.

In [ ]:
print(sales.groupby("region")["revenue"].sum().head(1))

*What's wrong:* 

*Your fix:*

> ### ✅ Answer B2
> **Symptom:** prints East (66000) — which happens to be correct here, but **by accident**.
> **Cause:** groupby sorts by **group key**, alphabetically. `head(1)` gives the first
> *alphabetically*, not the largest. Change one number and the answer silently becomes wrong.

In [ ]:
totals = sales.groupby("region")["revenue"].sum()

print("as returned (alphabetical):")
print(totals)
print()
print("sorted by VALUE:")
print(totals.sort_values(ascending=False))
print()
print("the actual top region:", totals.idxmax(), totals.max())

# Proof that head(1) is unreliable - shrink East and watch it still come first:
rigged = totals.copy()
rigged["East"] = 100
print()
print("with East reduced to 100, head(1) still says:", rigged.head(1).index[0])
print("but the real top is:", rigged.idxmax())

### B3. This should give the mean units and the max revenue.

In [ ]:
print(sales.groupby("region")["units"].agg({"units": "mean", "revenue": "max"}))

*Error:* 

*Your fix:*

> ### ✅ Answer B3
> **Error:** raises — you cannot pass a dictionary of *columns* after you have already
> narrowed the groupby to a single column.
> **Cause:** `["units"]` selects one column, so there is no `revenue` left to aggregate.

In [ ]:
# Option 1 - aggregate the whole groupby, not one column
print(sales.groupby("region").agg({"units": "mean", "revenue": "max"}))
print()

# Option 2 - named aggregation, which is clearer still
print(sales.groupby("region").agg(
    avg_units  = ("units",   "mean"),
    max_revenue= ("revenue", "max"),
))

# THE RULE: select the column when you want ONE function on ONE column.
# Aggregate the whole groupby when different columns need different functions.

### B4. This should compare each rep to their region's average.

In [ ]:
avg = sales.groupby("region")["units"].mean()
sales["region_avg"] = avg
print(sales[["rep", "region", "units", "region_avg"]])

*What's wrong:* 

*Your fix:*

> ### ✅ Answer B4
> **Symptom:** `region_avg` is full of `NaN`, with no error.
> **Cause:** `avg` has 3 rows indexed by region name (`East`, `North`, `West`). `sales` has
> 8 rows indexed 0–7. pandas aligns on the **index**, finds nothing in common, and fills
> everything with `NaN`.
>
> `agg` shrinks the table; you needed something that keeps the original shape.

In [ ]:
d = sales.copy()
d["region_avg"] = d.groupby("region")["units"].transform("mean").round(2)
print(d[["rep", "region", "units", "region_avg"]])
print()
print("above their own region average:",
      d[d["units"] > d["region_avg"]]["rep"].tolist())

# transform returns one value PER ROW, already aligned with the original index.
#
# The alternative is a merge, which you meet tomorrow:
#   sales.merge(avg.rename("region_avg"), on="region")

---
# Part C — Write the Code  *(12 min)*

---
# Part C — Write the Code

**Teaching note:** C5 is the one to demonstrate — two grouping keys plus unstack is the
shape of most real cross-tab reports.

### C1. Totals per region
Print the total revenue and total units for each region, sorted by revenue with the highest
first.

In [ ]:
# your code here

In [ ]:
totals = (sales.groupby("region")[["revenue", "units"]].sum()
               .sort_values("revenue", ascending=False))
print(totals)

#         revenue  units
# region
# East      66000    330
# West      52000    260
# North     38000    190
#
# Remember to sort - groupby returns alphabetical order, which would have
# put East first here by luck and North second, which is wrong.

### C2. A named report
Produce one row per region with: the number of reps, average units, total revenue and the
best single revenue figure. Give each column a readable name.

In [ ]:
# your code here

In [ ]:
report = sales.groupby("region").agg(
    reps        = ("rep",     "size"),
    avg_units   = ("units",   "mean"),
    total_rev   = ("revenue", "sum"),
    best_sale   = ("revenue", "max"),
).round(1)

print(report)

#         reps  avg_units  total_rev  best_sale
# region
# East       3      110.0      66000      28000
# North      2       95.0      38000      22000
# West       3       86.7      52000      24000
#
# Named aggregation keeps the output readable. Compare this with the
# column names you would get from agg(["size","mean","sum","max"]).

### C3. Count the rows honestly
One revenue figure is missing in the copy below. For each region print both the number of
rows and the number of revenue values present, and say which region has a gap.

In [ ]:
d = sales.copy()
d.loc[4, "revenue"] = np.nan        # Kiran, North

# your code here

In [ ]:
d = sales.copy()
d.loc[4, "revenue"] = np.nan        # Kiran, North

check = d.groupby("region").agg(
    rows      = ("rep",     "size"),
    reported  = ("revenue", "count"),
)
check["missing"] = check["rows"] - check["reported"]
print(check)

#         rows  reported  missing
# region
# East       3         3        0
# North      2         1        1
# West       3         3        0
#
# North has the gap. Its average revenue is now computed from ONE value,
# which is a very different claim from an average over two - and nothing
# in a plain .mean() would have told you.
print()
print("North average revenue:", d.groupby("region")["revenue"].mean()["North"],
      "from", check.loc["North", "reported"], "of", check.loc["North", "rows"], "reps")

### C4. Above your own group
Add a column showing each rep's region average units, then print the names of reps who beat
it.

In [ ]:
# your code here

In [ ]:
d = sales.copy()
d["region_avg"] = d.groupby("region")["units"].transform("mean").round(2)

print(d[["rep", "region", "units", "region_avg"]])
print()
print("beat their region average:", d[d["units"] > d["region_avg"]]["rep"].tolist())

# ['Ravi', 'Sara', 'Neha', 'Priya', 'Meena']
#
# Sara counts: West's average is 86.67 (120, 95, 45), and her 95 clears it.
#
# transform, not agg - the question is about ROWS that need to know
# something about their group, so the result must stay the original shape.

### C5. A cross-tab
Produce a table with regions as rows, products as columns, and total revenue in the cells.
Fill any empty combinations with 0.

In [ ]:
# your code here

In [ ]:
# Option 1 - groupby two keys, then unstack
cross = sales.groupby(["region", "product"])["revenue"].sum().unstack(fill_value=0)
print(cross)

# Option 2 - pivot_table, the spreadsheet way of thinking
print()
print(sales.pivot_table(index="region", columns="product",
                        values="revenue", aggfunc="sum", fill_value=0))

# product      A      B
# region
# East     38000  28000
# North    16000  22000
# West     24000  28000
#
# Both give exactly the same answer. Grouping by TWO keys produces a
# MultiIndex; unstack moves the last level up into the columns.

---
# Part D — Challenge  *(3 min, or take home)*

---
# Part D — Challenge

**Teaching note:** D1 is the audit habit. If students take one thing from today, make it the
assert.

### D1. A report that cannot lie
The data below has a missing region. Produce a per-region report of headcount and total
revenue that **accounts for every row**, and prove it with an assertion.

In [ ]:
d = sales.copy()
d.loc[0, "region"] = np.nan        # Ravi has no region

# your code here

In [ ]:
d = sales.copy()
d.loc[0, "region"] = np.nan        # Ravi has no region

# ---- the naive version, which quietly loses Ravi
naive = d.groupby("region").agg(headcount=("rep", "size"))
print("naive total:", naive["headcount"].sum(), "of", len(d), "rows")
print()

# ---- the honest version
report = d.groupby("region", dropna=False).agg(
    headcount = ("rep",     "size"),
    total_rev = ("revenue", "sum"),
)
assert report["headcount"].sum() == len(d), "some rows were dropped!"
print(report)
print()
print("accounts for every row:", report["headcount"].sum() == len(d))

# naive total: 7 of 8 rows       <- Ravi silently gone
#
#         headcount  total_rev
# region
# East            3      66000
# North           2      38000
# West            2      28000
# NaN             1      24000
#
# TWO things make this honest:
#   dropna=False  - the missing region becomes a visible NaN row
#   the assert    - turns a silent wrong answer into a loud failure
#
# Day 11's exceptions, used properly. An assert that fails is far better
# than a report that is quietly incomplete.

### D2. Rank within each group
For each region, find the **top-selling rep by revenue**, and show their share of that
region's total. *(Hint: `transform` gives you the group total per row.)*

In [ ]:
# your code here

In [ ]:
d = sales.copy()

# each rep's share of their own region's revenue
d["region_total"] = d.groupby("region")["revenue"].transform("sum")
d["share"] = (d["revenue"] / d["region_total"] * 100).round(1)

# the top rep in each region
top = d.loc[d.groupby("region")["revenue"].idxmax()]
print(top[["region", "rep", "revenue", "region_total", "share"]]
      .sort_values("region"))

#   region    rep  revenue  region_total  share
#     East   Neha    28000         66000   42.4
#    North  Priya    22000         38000   57.9
#     West   Ravi    24000         52000   46.2
#
# HOW IT WORKS:
#   groupby(...).idxmax()  gives the INDEX LABEL of the biggest row in each
#     group - not the value. Day 13's argmax idea, returning a label.
#   .loc[those labels]     pulls those whole rows out of the original table.
#   transform("sum")       puts each region's total beside every row so the
#     share can be computed without a merge.
#
# Priya's 57.9% means North depends heavily on one rep - the kind of finding
# a plain total would never surface.

---
## Done? Self-check

- [ ] I can explain split-apply-combine in my own words
- [ ] I know the group key becomes the **index** of the result
- [ ] I know results are sorted by key, not by value
- [ ] I know the difference between `size()` and `count()`
- [ ] I can write a named aggregation with four outputs
- [ ] I know when to use `transform` instead of `agg`
- [ ] I checked that my grouped counts add up to the row count

### Homework
1. Rewrite Day 15's loop-over-cities as a single `groupby`.
2. Build a report with four named aggregations and an `assert` that the counts add up.
3. Use `transform` to find every row above its own group's average.

### Next class — Topic 1.17: Merge, join & concat
Combining two tables on a shared key — and what happens to the rows that do not match.

---
*Slides & notebooks by Srinivasa Sai Chava · Boston University*

---
## Wrap-up — running the last 5 minutes

Three cold-call questions:

1. *"After a groupby, why does `result["region"]` fail?"* → it is the index, not a column.
2. *"`size()` versus `count()`?"* → rows versus non-missing values.
3. *"`agg` or `transform` for 'above your own group's average'?"* → `transform`.

**Common misconceptions to watch for today**

| Misconception | Correction |
|---|---|
| "The result is sorted biggest first" | Sorted by **group key**. Sort it yourself |
| "The group key is a column" | It is the **index**. `reset_index()` to change that |
| "`size` and `count` are the same" | Rows versus non-missing values |
| "`transform` gives me a summary" | It returns the **original shape** |
| "Every row ends up in some group" | Rows with a missing key are **dropped silently** |
| "`head(1)` gives the top group" | It gives the first **alphabetically** |

**B2 is worth doing live.** It prints the right answer by coincidence, then the rigged
version shows `head(1)` still naming East after East has been reduced to 100. A test that
passes for the wrong reason is more dangerous than one that fails.

**D1 is the habit to send them away with.** Two lines — `dropna=False` and an `assert` —
convert a report that can quietly lose rows into one that cannot. This is the third time
this week that missing data has hidden itself, and the students who start checking counts
now will stop losing marks on the capstone.

**Homework given:** rewrite the Day 15 loop as a groupby; a four-column named report with an
assert; `transform` to find above-group-average rows.

**Next session:** Topic 1.17 — merge, join and concat. Today reshaped one table; tomorrow
combines two, and the interesting part is what happens to rows that do not match.